# Blok 10: Praktyczna analiza danych z wykorzystaniem Pythona

Ten notebook zawiera rozwiązania krok po kroku. Możesz go traktować jako wzór „jak to dowieźć” w pracy.

**Zasada bloku:** pytanie biznesowe → wykres → 1 zdanie wniosku.


## Opis datasetów (skrót)

### 1) air_quality.csv (główny)
Pomiary godzinowe jakości powietrza dla wielu stacji.

Kolumny:
- `date_time` (str → datetime): czas pomiaru
- `station` (kategoria): nazwa stacji
- Zanieczyszczenia (float): `PM2.5`, `PM10`, `SO2`, `NO2`, `CO`, `O3`
- Pogoda/warunki (float/kategoria): `TEMP`, `PRES`, `DEWP`, `RAIN`, `WSPM`, `wd`

### 2) bank.csv
Dane kampanii bankowej, target `deposit` (yes/no).

Kolumny (wybrane):
- numeryczne: `age`, `balance`, `duration`, `campaign`, `pdays`, `previous`
- kategoryczne: `job`, `education`, `housing`, `loan`, `poutcome`, `month`, `deposit`

### 3) AI_Poland_Jobs_2022_06_10.csv (rozgrzewka)
Ogłoszenia o pracę.

Kolumny:
- `Title`, `Senior` (bool), `Company`, `Location`
- `Salary_Min`, `Salary_Max` (int)


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

plt.rcParams["figure.figsize"] = (10, 4)
sns.set_theme(style="whitegrid")

print("Imports OK")


In [ ]:
# Jeśli masz pliki w ./data, zmień na "data/air_quality.csv" itd.
PATH_AIR = "air_quality.csv"
PATH_BANK = "bank.csv"
PATH_JOBS = "AI_Poland_Jobs_2022_06_10.csv"


In [ ]:
df_air = pd.read_csv(PATH_AIR)
df_bank = pd.read_csv(PATH_BANK)
df_jobs = pd.read_csv(PATH_JOBS)

for name, df in [("air", df_air), ("bank", df_bank), ("jobs", df_jobs)]:
    print(name, df.shape)
    display(df.head(3))


## Rozgrzewka (Dataset C: AI jobs)


### Zadanie 1
**Biznes:** Ile ogłoszeń mamy w podziale na `Senior`?

**Rozwiązanie:** `value_counts` + udział procentowy.


In [ ]:
counts = df_jobs["Senior"].value_counts(dropna=False)
share = (df_jobs["Senior"].value_counts(normalize=True, dropna=False) * 100).round(1)

result = pd.DataFrame({"count": counts, "share_pct": share})
result.index.name = "Senior"
display(result)


### Zadanie 2
**Biznes:** Top 10 lokalizacji po liczbie ofert.


In [ ]:
top_loc = df_jobs["Location"].value_counts().head(10)
display(top_loc.to_frame("offers"))


### Zadanie 3
**Biznes:** Jak wyglądają widełki płac? (Salary_Min vs Salary_Max)

**Rozwiązanie:** dodajemy `salary_mid` (środek widełek) i rysujemy histogram.


In [ ]:
df_jobs = df_jobs.copy()
df_jobs["salary_mid"] = (df_jobs["Salary_Min"] + df_jobs["Salary_Max"]) / 2

sns.histplot(df_jobs["salary_mid"], kde=True, bins=20)
plt.title("Rozkład salary_mid (AI jobs)")
plt.xlabel("salary_mid")
plt.ylabel("Liczba ofert")
plt.tight_layout()
plt.show()


### Zadanie 4
**Biznes:** Czy oferty `Senior=True` mają wyższe widełki?

**Rozwiązanie:** boxplot `salary_mid` per `Senior`.


In [ ]:
sns.boxplot(data=df_jobs, x="Senior", y="salary_mid")
plt.title("Salary_mid vs Senior")
plt.xlabel("Senior")
plt.ylabel("salary_mid")
plt.tight_layout()
plt.show()


**Wniosek:** Oferty `Senior=True` mają zwykle wyższy poziom `salary_mid` (medianę i górny kwartyl) niż oferty nieseniorskie — ale rozkłady mogą się częściowo nakładać.


## Dataset A: air_quality — EDA + Matplotlib/Seaborn


### Zadanie 5
**Biznes:** Sanity check: typy, braki, statystyki.


In [ ]:
df_air.info()

missing_top = df_air.isna().sum().sort_values(ascending=False).head(10)
print("Top braki:\n", missing_top)

display(df_air.describe().T.head(12))


**Obserwacje (przykład):**
- W danych występują braki w części kolumn pomiarowych/pogodowych — przed analizą trzeba je policzyć i zadecydować, czy ignorujemy, imputujemy, czy agregujemy.
- Skale niektórych zmiennych (np. `CO`, `RAIN`) mogą mieć outliery — warto sprawdzić kwantyle i nie robić wniosków tylko po średniej.


### Zadanie 6
**Biznes:** Datetime + sortowanie (podstawa analizy czasowej).


In [ ]:
df_air = df_air.copy()
df_air["date_time"] = pd.to_datetime(df_air["date_time"], errors="coerce")
df_air = df_air.sort_values(["date_time", "station"])

print("NaT in date_time:", df_air["date_time"].isna().sum())
display(df_air[["date_time", "station"]].head())


### Zadanie 7
**Biznes:** Zakres czasu i liczba stacji.


In [ ]:
start = df_air["date_time"].min()
end = df_air["date_time"].max()
n_stations = df_air["station"].nunique()
stations = sorted(df_air["station"].unique().tolist())

print(f"Zakres: {start} → {end}")
print(f"Liczba stacji: {n_stations}")
print("Stacje:", stations)


### Zadanie 8
**Biznes:** Rozkład PM2.5 i outliery (p95/p99).


In [ ]:
sns.histplot(df_air["PM2.5"].dropna(), kde=True, bins=40)
plt.title("PM2.5 — rozkład (całość)")
plt.xlabel("PM2.5 [µg/m³]")
plt.ylabel("Liczba obserwacji")
plt.tight_layout()
plt.show()

p95 = df_air["PM2.5"].quantile(0.95)
p99 = df_air["PM2.5"].quantile(0.99)
print("PM2.5 p95:", round(p95, 2))
print("PM2.5 p99:", round(p99, 2))


### Zadanie 9
**Biznes:** Które stacje mają najwyższe typowe PM2.5? (mediana)


In [ ]:
med = df_air.groupby("station")["PM2.5"].median().sort_values(ascending=False)

sns.barplot(x=med.values, y=med.index)
plt.title("PM2.5 — mediana per stacja")
plt.xlabel("PM2.5 [µg/m³]")
plt.ylabel("Stacja")
plt.tight_layout()
plt.show()

display(med.head(10).to_frame("pm25_median"))


### Zadanie 10
**Biznes:** Trend w czasie — agregacja dzienna (wszystkie stacje razem).


In [ ]:
df_air["date"] = df_air["date_time"].dt.date

df_daily = (
    df_air.groupby("date", as_index=False)
    .agg({"PM2.5": "mean", "PM10": "mean"})
)
display(df_daily.head())


### Zadanie 11
**Biznes:** Pokaż trend dzienny PM2.5 jako linię.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_daily["date"], df_daily["PM2.5"])
ax.set_title("PM2.5 — średnia dzienna (wszystkie stacje)")
ax.set_xlabel("Data")
ax.set_ylabel("PM2.5 [µg/m³]")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Zadanie 12
**Biznes:** Rolling mean 7 dni — wygładzenie trendu.


In [ ]:
df_daily = df_daily.sort_values("date").copy()
df_daily["pm25_roll7"] = df_daily["PM2.5"].rolling(window=7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_daily["date"], df_daily["PM2.5"], label="PM2.5 (daily mean)", alpha=0.6)
ax.plot(df_daily["date"], df_daily["pm25_roll7"], label="Rolling mean 7d", linewidth=2)
ax.set_title("PM2.5 — średnia dzienna + rolling mean 7d")
ax.set_xlabel("Data")
ax.set_ylabel("PM2.5 [µg/m³]")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


### Zadanie 13
**Biznes:** Sezonowość — PM2.5 w miesiącach.


In [ ]:
df_air["month"] = df_air["date_time"].dt.month

m = df_air.groupby("month")["PM2.5"].median().reindex(range(1, 13))

sns.barplot(x=m.index, y=m.values)
plt.title("PM2.5 — mediana per miesiąc")
plt.xlabel("Miesiąc")
plt.ylabel("PM2.5 [µg/m³]")
plt.tight_layout()
plt.show()


**Wniosek:** PM2.5 wykazuje sezonowość — typowo wyższe wartości wypadają w miesiącach zimowych, a niższe w cieplejszych (dokładny kształt zależy od danych i zakresu czasu).


### Zadanie 14
**Biznes:** Wzorzec godzinowy — czy w nocy jest gorzej?


In [ ]:
df_air["hour"] = df_air["date_time"].dt.hour
h = df_air.groupby("hour")["PM2.5"].median().reindex(range(24))

plt.plot(h.index, h.values)
plt.title("PM2.5 — mediana po godzinie")
plt.xlabel("Godzina")
plt.ylabel("PM2.5 [µg/m³]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Zadanie 15
**Biznes:** Korelacje (uwaga: to nie jest przyczynowość).


In [ ]:
num_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
corr = df_air[num_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=False, center=0)
plt.title("Korelacje — air_quality")
plt.tight_layout()
plt.show()


### Zadanie 16
**Biznes:** Relacja TEMP vs PM2.5.

**Rozwiązanie:** próbka (żeby nie mielić 95k punktów) + scatter.


In [ ]:
df_sample = df_air.dropna(subset=["TEMP", "PM2.5"]).sample(n=5000, random_state=42)
sns.scatterplot(data=df_sample, x="TEMP", y="PM2.5", alpha=0.3)
plt.title("TEMP vs PM2.5 (próba 5000)")
plt.xlabel("TEMP")
plt.ylabel("PM2.5 [µg/m³]")
plt.tight_layout()
plt.show()

# Opcjonalnie: trendline (pamiętaj: to nie przyczynowość)
sns.regplot(data=df_sample, x="TEMP", y="PM2.5", scatter_kws={"alpha": 0.2}, line_kws={"color": "black"})
plt.title("TEMP vs PM2.5 + trendline (próba)")
plt.tight_layout()
plt.show()


### Zadanie 17
**Biznes:** Ranking stacji zimą (miesiące 12,1,2) — top 8.


In [ ]:
winter = df_air[df_air["month"].isin([12, 1, 2])].copy()
rank = winter.groupby("station")["PM2.5"].median().sort_values(ascending=False)
top8 = rank.head(8)

sns.barplot(x=top8.values, y=top8.index)
plt.title("Top 8 stacji — PM2.5 (mediana, zima)")
plt.xlabel("PM2.5 [µg/m³]")
plt.ylabel("Stacja")
plt.tight_layout()
plt.show()

display(top8.to_frame("pm25_median_winter"))


### Zadanie 18
**Biznes:** Porównaj dwie stacje w czasie (np. najgorsza i najlepsza z top 8).


In [ ]:
s1 = top8.index[0]
s2 = top8.index[-1]

df2 = df_air[df_air["station"].isin([s1, s2])].copy()
df2["date"] = df2["date_time"].dt.date

df2_daily = df2.groupby(["date", "station"], as_index=False)["PM2.5"].mean()

fig, ax = plt.subplots(figsize=(10, 4))
for st in [s1, s2]:
    tmp = df2_daily[df2_daily["station"] == st]
    ax.plot(tmp["date"], tmp["PM2.5"], label=st)

ax.set_title("PM2.5 — trend dzienny (2 stacje)")
ax.set_xlabel("Data")
ax.set_ylabel("PM2.5 [µg/m³]")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


## Plotly (Dataset A) — interaktywność + eksport do HTML


### Zadanie 19
**Biznes:** Interaktywny trend dzienny PM2.5 (all stations) + zapis do HTML.


In [ ]:
fig = px.line(df_daily, x="date", y="PM2.5", title="PM2.5 — średnia dzienna (all stations)")
fig.update_layout(xaxis_title="Data", yaxis_title="PM2.5 [µg/m³]")
fig.write_html("plot_pm25_daily.html")
fig.show()


### Zadanie 20
**Biznes:** Trend PM2.5 per stacja (top 5) — interaktywnie.


In [ ]:
top5_stations = med.head(5).index.tolist()

df_top = df_air[df_air["station"].isin(top5_stations)].copy()
df_top["date"] = df_top["date_time"].dt.date
df_top_daily = df_top.groupby(["date", "station"], as_index=False)["PM2.5"].mean()

fig = px.line(
    df_top_daily,
    x="date",
    y="PM2.5",
    color="station",
    title="PM2.5 — trend dzienny (Top 5 stacji)"
)
fig.update_layout(xaxis_title="Data", yaxis_title="PM2.5 [µg/m³]")
fig.write_html("plot_pm25_top5_stations.html")
fig.show()


### Zadanie 21
**Biznes:** Rozkłady PM2.5 między stacjami (top 8) — interaktywnie.


In [ ]:
df_top8 = df_air[df_air["station"].isin(top8.index)].copy()

fig = px.box(df_top8, x="station", y="PM2.5", points=False, title="PM2.5 — rozkład (Top 8 stacji)")
fig.update_layout(xaxis_title="Stacja", yaxis_title="PM2.5 [µg/m³]")
fig.write_html("plot_pm25_box_top8.html")
fig.show()


### Zadanie 22
**Biznes:** TEMP vs PM2.5 z kolorowaniem po sezonie.


In [ ]:
def month_to_season(m: int) -> str:
    if m in [12, 1, 2]:
        return "winter"
    if m in [3, 4, 5]:
        return "spring"
    if m in [6, 7, 8]:
        return "summer"
    return "autumn"

df_air["season"] = df_air["month"].apply(month_to_season)

df_sample2 = df_air.dropna(subset=["TEMP", "PM2.5"]).sample(n=8000, random_state=42)

fig = px.scatter(
    df_sample2,
    x="TEMP",
    y="PM2.5",
    color="season",
    opacity=0.45,
    title="TEMP vs PM2.5 (kolor: sezon)"
)
fig.update_layout(xaxis_title="TEMP", yaxis_title="PM2.5 [µg/m³]")
fig.write_html("plot_temp_vs_pm25_season.html")
fig.show()


### Zadanie 23
**Biznes:** Wykres braków danych per kolumna (do raportu).


In [ ]:
missing_pct = (df_air.isna().mean() * 100).sort_values(ascending=False)
miss_df = missing_pct.reset_index()
miss_df.columns = ["column", "missing_pct"]

fig = px.bar(miss_df, x="missing_pct", y="column", orientation="h", title="Braki danych [%] — air_quality")
fig.update_layout(xaxis_title="Missing [%]", yaxis_title="Column")
fig.write_html("plot_missing_air_quality.html")
fig.show()


### Zadanie 24
**Biznes:** Mini-dashboard: trend dzienny + rozkład w jednym HTML.


In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=("Trend dzienny", "Rozkład PM2.5"))

# Trend dzienny
fig.add_trace(
    go.Scatter(x=df_daily["date"], y=df_daily["PM2.5"], mode="lines", name="PM2.5 daily"),
    row=1, col=1
)

# Histogram (na surowych danych, ale bez NaN)
pm25_vals = df_air["PM2.5"].dropna()
fig.add_trace(
    go.Histogram(x=pm25_vals, nbinsx=60, name="PM2.5 hist"),
    row=1, col=2
)

fig.update_layout(title_text="Air Quality — mini-dashboard", showlegend=False)
fig.update_xaxes(title_text="Data", row=1, col=1)
fig.update_yaxes(title_text="PM2.5 [µg/m³]", row=1, col=1)
fig.update_xaxes(title_text="PM2.5 [µg/m³]", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.write_html("dashboard_air_quality.html")
fig.show()


## Dataset B: bank — wizualizacje pod decyzję kampanijną


### Zadanie 25
**Biznes:** Overall conversion rate (`deposit=yes`) + wykres udziałów.


In [ ]:
conv = df_bank["deposit"].value_counts(normalize=True) * 100
conv_df = conv.rename("share_pct").reset_index().rename(columns={"index": "deposit"})
display(conv_df)

sns.barplot(data=conv_df, x="deposit", y="share_pct")
plt.title("Deposit — udział klas")
plt.xlabel("deposit")
plt.ylabel("Udział [%]")
plt.tight_layout()
plt.show()


### Zadanie 26
**Biznes:** Które zawody (`job`) mają najwyższy conversion?


In [ ]:
rate = (
    df_bank.assign(deposit_yes=(df_bank["deposit"] == "yes"))
    .groupby("job")["deposit_yes"].mean()
)
top10 = rate.sort_values(ascending=False).head(10)
display(top10.to_frame("conversion_rate"))

sns.barplot(x=top10.values, y=top10.index)
plt.title("Top 10 job — conversion (deposit=yes)")
plt.xlabel("Conversion rate")
plt.ylabel("Job")
plt.tight_layout()
plt.show()


### Zadanie 27
**Biznes:** Czy `balance` różni się między `deposit=yes` i `deposit=no`?


In [ ]:
sns.boxplot(data=df_bank, x="deposit", y="balance")
plt.title("Balance vs deposit")
plt.xlabel("deposit")
plt.ylabel("balance")
plt.tight_layout()
plt.show()

med_balance = df_bank.groupby("deposit")["balance"].median()
display(med_balance.to_frame("median_balance"))


**Wniosek:** Zwykle widać różnicę w rozkładach `balance` między klasami `deposit` (sprawdź mediany i kwartyle) — to kandydat do segmentacji, ale pamiętaj o outlierach.


### Zadanie 28
**Biznes:** Jak `duration` rozmowy wpływa na wynik?

Uwaga: to cecha „po fakcie”, więc w modelowaniu to leakage, ale analitycznie bywa użyteczna.


In [ ]:
sns.violinplot(data=df_bank, x="deposit", y="duration", cut=0)
plt.title("Duration vs deposit")
plt.xlabel("deposit")
plt.ylabel("duration")
plt.tight_layout()
plt.show()


## Raportowanie — mini-projekt (Excel)

**Deliverable:** `report.xlsx` (obowiązkowo) + HTML z Plotly (już zapisane).


### Zadanie 29
**Biznes:** Tabela KPI do arkusza `summary`.

Zróbmy prosty zestaw: zakres dat, liczba stacji, średnie/mediany, top 3 stacje.


In [ ]:
date_start = df_air["date_time"].min()
date_end = df_air["date_time"].max()
n_st = df_air["station"].nunique()

pm25_mean = df_air["PM2.5"].mean()
pm25_median = df_air["PM2.5"].median()
pm10_mean = df_air["PM10"].mean()
pm10_median = df_air["PM10"].median()

top3_stations = med.head(3)
top3_str = ", ".join([f"{idx} ({val:.1f})" for idx, val in top3_stations.items()])

df_summary_kpi = pd.DataFrame(
    {
        "metric": [
            "Date range",
            "Stations (count)",
            "PM2.5 mean",
            "PM2.5 median",
            "PM10 mean",
            "PM10 median",
            "Top 3 stations by PM2.5 median",
        ],
        "value": [
            f"{date_start} → {date_end}",
            int(n_st),
            float(pm25_mean),
            float(pm25_median),
            float(pm10_mean),
            float(pm10_median),
            top3_str,
        ],
    }
)
display(df_summary_kpi)


### Zadanie 30
**Biznes:** `report.xlsx` z 2 arkuszami + formatowanie.

W `details` wrzucimy: top 8 stacji (mediana PM2.5) i miesięczne mediany PM2.5.

Bonus: zapis jednego wykresu do PNG i wstawienie do Excela.


In [ ]:
# 1) tabele szczegółowe
df_top8_table = top8.reset_index()
df_top8_table.columns = ["station", "pm25_median_winter"]

df_monthly = (
    df_air.groupby("month", as_index=False)
    .agg(pm25_median=("PM2.5", "median"), pm10_median=("PM10", "median"))
    .sort_values("month")
)

# 2) wykres PNG (do wstawienia w Excel)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(df_daily["date"], df_daily["pm25_roll7"], linewidth=2)
ax.set_title("PM2.5 — rolling mean 7d")
ax.set_xlabel("Date")
ax.set_ylabel("PM2.5 [µg/m³]")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("chart_pm25_roll7.png", dpi=150, bbox_inches="tight")
plt.show()

# 3) Excel
with pd.ExcelWriter("report.xlsx", engine="xlsxwriter") as writer:
    df_summary_kpi.to_excel(writer, sheet_name="summary", index=False)
    df_top8_table.to_excel(writer, sheet_name="details", index=False, startrow=0)
    df_monthly.to_excel(writer, sheet_name="details", index=False, startrow=len(df_top8_table) + 3)

    workbook = writer.book
    ws_sum = writer.sheets["summary"]
    ws_det = writer.sheets["details"]

    fmt_header = workbook.add_format({"bold": True, "bg_color": "#EEEEEE"})
    fmt_num = workbook.add_format({"num_format": "0.00"})

    # nagłówki
    ws_sum.set_row(0, None, fmt_header)
    ws_det.set_row(0, None, fmt_header)

    # freeze panes
    ws_sum.freeze_panes(1, 0)
    ws_det.freeze_panes(1, 0)

    # szerokości kolumn
    ws_sum.set_column(0, 0, 28)
    ws_sum.set_column(1, 1, 48)

    ws_det.set_column(0, 0, 22)
    ws_det.set_column(1, 3, 18)

    # format liczb (podstawowy przykład: w summary wiersze 2-5 w kolumnie value)
    # Uwaga: KPI są mieszane (tekst + liczby), więc formatujemy ostrożnie.
    # Tutaj przykład: jeśli wiesz, gdzie są liczby, możesz to doprecyzować.
    for row in range(2, 6):
        ws_sum.write_number(row, 1, float(df_summary_kpi.loc[row, "value"]), fmt_num)

    # wstaw obrazek
    ws_sum.insert_image("D2", "chart_pm25_roll7.png")

print("Zapisano report.xlsx")


### Bonus (opcjonalnie): esparto / folium

- Jeśli chcesz: zrób prosty raport HTML w esparto (sekcje + obrazy wykresów).
- Folium ma sens, jeśli masz współrzędne stacji lub tabelę mapującą `station → (lat, lon)`.

W tym bloku **Excel + HTML z Plotly** jest w pełni akceptowalnym „produkcyjnym” rezultatem.
